In [9]:
import numpy as np
import pandas as pd


In [10]:
pair = 'ETH_USDT'
timeframe = '8h'

exchange = 'binance'
def load_candles(exchange, pair, timeframe):
    odf = pd.read_json(f'/media/mu6mula/Data/Crypto-Data-Feed/freq-user-data/data/{exchange}/{pair}-{timeframe}.json'
    # exchange = 'kucoin'
    # odf = pd.read_json(f'../../freq-user-data/data/{exchange}/futures/{pair}-{timeframe}-futures.json'
    ).dropna().set_axis(['timestamp', 'open', 'high', 'low', 'close', 'volume'], axis=1
    ).assign(dtime=lambda x: pd.to_datetime(x['timestamp'], unit='ms', utc=False)
    ).set_index('dtime').sort_index()
    return odf

# print(odf.shape)
odf = load_candles(exchange, pair, timeframe)
odf.head(4)

,timestamp,open,high,low,close,volume
dtime,,,,,,
2020-01-01 00:00:00,1577836800000,129.16,130.98,128.68,130.24,47143.32874
2020-01-01 08:00:00,1577865600000,130.24,132.40,129.87,132.08,51833.22852
2020-01-01 16:00:00,1577894400000,132.08,133.05,129.74,130.77,45793.96471
2020-01-02 00:00:00,1577923200000,130.72,130.78,128.69,129.26,66066.00891


In [11]:
def annualized_return(returns):
    cumulative_return = (1 + returns).prod() - 1
    num_periods = len(returns)
    annualized_return = (1 + cumulative_return) ** (252 / num_periods) - 1
    return annualized_return

def annualized_volatility(returns):
    volatility = returns.std() * (252 ** 0.5)
    return volatility

def sharpe_ratio(returns, risk_free_rate=0):
    excess_return = returns - risk_free_rate / 252
    annual_return = annualized_return(returns)
    annual_volatility = annualized_volatility(returns)
    sharpe_ratio = annual_return / annual_volatility
    return sharpe_ratio

def max_drawdown(returns):
    cumulative_returns = (1 + returns).cumprod()
    peak = cumulative_returns.cummax()
    drawdown = (cumulative_returns - peak) / peak
    max_drawdown = drawdown.min()
    return max_drawdown

def win_rate(returns):
    wins = returns[returns > 0].count()
    total_trades = returns.count()
    win_rate = wins / total_trades
    return win_rate

def calculate_detailed_metrics(df):
    total_return = df['cumulative_returns'].iloc[-1] - 1
    annual_return = annualized_return(df['strategy_returns'])
    annual_volatility = annualized_volatility(df['strategy_returns'])
    sharpe = sharpe_ratio(df['strategy_returns'])
    drawdown = max_drawdown(df['strategy_returns'])
    winrate = win_rate(df['strategy_returns'])
    num_trades = len(df[df['signal'] != 0])
    avg_profit = df[df['strategy_returns'] > 0]['strategy_returns'].mean()
    avg_loss = df[df['strategy_returns'] < 0]['strategy_returns'].mean()

    return {
        'Total Return': total_return,
        'Annualized Return': annual_return,
        'Annualized Volatility': annual_volatility,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': drawdown,
        'Win Rate': winrate,
        'Number of Trades': num_trades,
        'Average Profit': avg_profit,
        'Average Loss': avg_loss
    }


In [12]:
df = odf

In [15]:
# Function to calculate ATR
def calculate_atr(df, window=14):
    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    tr = high_low.combine(high_close, max).combine(low_close, max)
    atr = tr.rolling(window=window, min_periods=1).mean()
    return atr

# Function to calculate RSI
def calculate_rsi(df, period=14):
    delta = df['close'].diff(1)
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.rolling(window=period, min_periods=1).mean()
    avg_loss = loss.rolling(window=period, min_periods=1).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

# Function to generate trend signals with dual EMA and RSI
def generate_trend_signals(df, short_ema_period, long_ema_period, rsi_period, rsi_oversold, rsi_overbought):
    df['short_EMA'] = df['close'].ewm(span=short_ema_period, adjust=False).mean()
    df['long_EMA'] = df['close'].ewm(span=long_ema_period, adjust=False).mean()
    df['RSI'] = calculate_rsi(df, rsi_period)
    df['trend_signal'] = np.where((df['short_EMA'] > df['long_EMA']) & (df['RSI'] < rsi_overbought), 1, 0)
    df['trend_signal'] = np.where((df['short_EMA'] < df['long_EMA']) & (df['RSI'] > rsi_oversold), -1, df['trend_signal'])
    return df

# Function to backtest strategy with dynamic allocation based on volatility
def backtest_dynamic_allocation(df, look_back_period, atr_period=14, stop_loss_mult=2, take_profit_mult=2, max_alloc=1, min_alloc=0.1):
    df['log_price'] = np.log(df['close'])
    df['log_momentum'] = df['log_price'].diff(look_back_period)
    df['ATR'] = calculate_atr(df, atr_period)
    df['signal'] = np.where((df['log_momentum'] > 0) & (df['trend_signal'] == 1), 1,
                            np.where((df['log_momentum'] < 0) & (df['trend_signal'] == -1), -1, 0))

    df['strategy_returns'] = 0
    trades = []
    position_open = False
    entry_price = 0

    for i in range(1, len(df)):
        atr_value = df['ATR'].iloc[i-1]
        allocation = max(min_alloc, min(max_alloc, 1 / atr_value))
        if not position_open:
            if df['signal'].iloc[i-1] == 1:
                entry_price = df['close'].iloc[i-1]
                position_open = True
                position_type = 'long'
            elif df['signal'].iloc[i-1] == -1:
                entry_price = df['close'].iloc[i-1]
                position_open = True
                position_type = 'short'
        else:
            if position_type == 'long':
                stop_loss = entry_price - stop_loss_mult * atr_value
                take_profit = entry_price + take_profit_mult * atr_value
                if df['low'].iloc[i] < stop_loss:
                    df['strategy_returns'].iloc[i] = allocation * (stop_loss - entry_price) / entry_price
                    trades.append((stop_loss - entry_price) / entry_price)
                    position_open = False
                elif df['high'].iloc[i] > take_profit:
                    df['strategy_returns'].iloc[i] = allocation * (take_profit - entry_price) / entry_price
                    trades.append((take_profit - entry_price) / entry_price)
                    position_open = False
                else:
                    df['strategy_returns'].iloc[i] = allocation * df['returns'].iloc[i]
            elif position_type == 'short':
                stop_loss = entry_price + stop_loss_mult * atr_value
                take_profit = entry_price - take_profit_mult * atr_value
                if df['high'].iloc[i] > stop_loss:
                    df['strategy_returns'].iloc[i] = allocation * (entry_price - stop_loss) / entry_price
                    trades.append((entry_price - stop_loss) / entry_price)
                    position_open = False
                elif df['low'].iloc[i] < take_profit:
                    df['strategy_returns'].iloc[i] = allocation * (entry_price - take_profit) / entry_price
                    trades.append((entry_price - take_profit) / entry_price)
                    position_open = False
                else:
                    df['strategy_returns'].iloc[i] = allocation * df['returns'].iloc[i]

    df['cumulative_returns'] = (1 + df['strategy_returns']).cumprod()
    total_return = df['cumulative_returns'].iloc[-1] - 1
    annual_return = annualized_return(df['strategy_returns'])
    annual_volatility = annualized_volatility(df['strategy_returns'])
    sharpe = sharpe_ratio(df['strategy_returns'])
    drawdown = max_drawdown(df['strategy_returns'])
    winrate = win_rate(df['strategy_returns'])
    num_trades = len(trades)
    avg_profit = np.mean([trade for trade in trades if trade > 0]) if trades else 0
    avg_loss = np.mean([trade for trade in trades if trade < 0]) if trades else 0

    return df, total_return, annual_return, annual_volatility, sharpe, drawdown, winrate, num_trades, avg_profit, avg_loss


In [16]:
# Apply the final strategy to the bear market data_bear_market
df_signals_final = generate_trend_signals(df.copy(), 20, 200, 14, 30, 70)
df_final_optimized, total_return, annual_return, annual_volatility, sharpe, drawdown, winrate, num_trades, avg_profit, avg_loss = backtest_dynamic_allocation(
    df_signals_final, best_look_back_period, atr_period=14, stop_loss_mult=2.0, take_profit_mult=2.0, max_alloc=1, min_alloc=0.1)

# Calculate detailed metrics
final_metrics = calculate_detailed_metrics(df_final_optimized)
final_metrics


NameError: name 'best_look_back_period' is not defined